In [1]:
import duckdb

In [4]:
duckdb.sql("""
    DESCRIBE
    SELECT *
    FROM 'february_cart_events.csv'
""")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_time    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ event_type    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ purchased     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

In [6]:
duckdb.sql("""
    SELECT
        COUNT(DISTINCT user_session) AS total_sessions,
        COUNT(DISTINCT CASE
            WHEN event_type = 'purchase' THEN user_session
        END) AS purchased_sessions,
        COUNT(DISTINCT CASE
            WHEN event_type = 'cart' THEN user_session
        END) AS cart_sessions,
        COUNT(DISTINCT CASE
            WHEN event_type = 'cart'
             AND user_session NOT IN (
                 SELECT DISTINCT user_session
                 FROM 'february_cart_events.csv'
                 WHERE event_type = 'purchase'
             )
            THEN user_session
        END) AS abandoned_sessions
    FROM 'february_cart_events.csv'
""")

┌────────────────┬────────────────────┬───────────────┬────────────────────┐
│ total_sessions │ purchased_sessions │ cart_sessions │ abandoned_sessions │
│     int64      │       int64        │     int64     │       int64        │
├────────────────┼────────────────────┼───────────────┼────────────────────┤
│        1660109 │                  0 │       1660109 │            1660109 │
└────────────────┴────────────────────┴───────────────┴────────────────────┘

In [7]:
duckdb.sql("""
    SELECT
        event_type,
        COUNT(*) AS event_count
    FROM 'february_cart_events.csv'
    GROUP BY event_type
    ORDER BY event_count DESC
""")

┌────────────┬─────────────┐
│ event_type │ event_count │
│  varchar   │    int64    │
├────────────┼─────────────┤
│ cart       │     2656529 │
└────────────┴─────────────┘

In [8]:
duckdb.sql("""
    SELECT
        purchased,
        COUNT(*) AS row_count,
        COUNT(DISTINCT user_session) AS sessions
    FROM 'february_cart_events.csv'
    GROUP BY purchased
    ORDER BY purchased
""")

┌───────────┬───────────┬──────────┐
│ purchased │ row_count │ sessions │
│  boolean  │   int64   │  int64   │
├───────────┼───────────┼──────────┤
│ false     │   1363052 │   960018 │
│ true      │   1293477 │   700091 │
└───────────┴───────────┴──────────┘

## Business Question 1: What is the overall purchase and cart abandonment rate?

In [9]:
duckdb.sql("""
    SELECT
        COUNT(DISTINCT user_session) AS total_sessions,
        COUNT(DISTINCT CASE
            WHEN purchased = TRUE THEN user_session
        END) AS purchased_sessions,
        COUNT(DISTINCT CASE
            WHEN purchased = FALSE THEN user_session
        END) AS abandoned_sessions,

        ROUND(
            100.0 * COUNT(DISTINCT CASE
                WHEN purchased = TRUE THEN user_session
            END)
            / COUNT(DISTINCT user_session),
            2
        ) AS conversion_rate,

        ROUND(
            100.0 * COUNT(DISTINCT CASE
                WHEN purchased = FALSE THEN user_session
            END)
            / COUNT(DISTINCT user_session),
            2
        ) AS abandonment_rate

    FROM 'february_cart_events.csv'
""")

┌────────────────┬────────────────────┬────────────────────┬─────────────────┬──────────────────┐
│ total_sessions │ purchased_sessions │ abandoned_sessions │ conversion_rate │ abandonment_rate │
│     int64      │       int64        │       int64        │     double      │      double      │
├────────────────┼────────────────────┼────────────────────┼─────────────────┼──────────────────┤
│        1660109 │             700091 │             960018 │           42.17 │            57.83 │
└────────────────┴────────────────────┴────────────────────┴─────────────────┴──────────────────┘

## Business Question 2: Does the number of carted products affect purchase likelihood?

In [10]:
duckdb.sql("""
    SELECT
        purchased,
        COUNT(DISTINCT user_session) AS sessions,
        ROUND(AVG(product_count), 2) AS avg_products_per_session
    FROM (
        SELECT
            user_session,
            purchased,
            COUNT(DISTINCT product_id) AS product_count
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    )
    GROUP BY purchased
    ORDER BY purchased
""")

┌───────────┬──────────┬──────────────────────────┐
│ purchased │ sessions │ avg_products_per_session │
│  boolean  │  int64   │          double          │
├───────────┼──────────┼──────────────────────────┤
│ false     │   960018 │                     1.09 │
│ true      │   700091 │                     1.21 │
└───────────┴──────────┴──────────────────────────┘

## Business Question 3: How does purchase behavior change with the number of distinct products in the cart?

In [12]:
duckdb.sql("""
    WITH session_products AS (
        SELECT
            user_session,
            purchased,
            COUNT(DISTINCT product_id) AS product_count
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    ),

    product_bands AS (
        SELECT
            user_session,
            purchased,
            CASE
                WHEN product_count = 1 THEN '1 product'
                WHEN product_count = 2 THEN '2 products'
                WHEN product_count = 3 THEN '3 products'
                ELSE '4+ products'
            END AS product_count_band,

            CASE
                WHEN product_count = 1 THEN 1
                WHEN product_count = 2 THEN 2
                WHEN product_count = 3 THEN 3
                ELSE 4
            END AS band_order

        FROM session_products
    )

    SELECT
        product_count_band,
        COUNT(*) AS sessions,
        SUM(CASE WHEN purchased = TRUE THEN 1 ELSE 0 END) AS purchased_sessions,

        ROUND(
            100.0 * SUM(CASE WHEN purchased = TRUE THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS purchase_rate

    FROM product_bands

    GROUP BY
        product_count_band,
        band_order

    ORDER BY band_order
""")

┌────────────────────┬──────────┬────────────────────┬───────────────┐
│ product_count_band │ sessions │ purchased_sessions │ purchase_rate │
│      varchar       │  int64   │       int128       │    double     │
├────────────────────┼──────────┼────────────────────┼───────────────┤
│ 1 product          │  1484011 │             591298 │         39.84 │
│ 2 products         │   139468 │              83927 │         60.18 │
│ 3 products         │    25332 │              16996 │         67.09 │
│ 4+ products        │    11299 │               7870 │         69.65 │
└────────────────────┴──────────┴────────────────────┴───────────────┘

## Business Question 4: Does product price relate to purchase behavior?

In [13]:
duckdb.sql("""
    SELECT
        MIN(price) AS min_price,
        ROUND(AVG(price), 2) AS avg_price,
        MEDIAN(price) AS median_price,
        MAX(price) AS max_price
    FROM 'february_cart_events.csv'
""")

┌───────────┬───────────┬──────────────┬───────────┐
│ min_price │ avg_price │ median_price │ max_price │
│  double   │  double   │    double    │  double   │
├───────────┼───────────┼──────────────┼───────────┤
│       0.0 │    291.55 │       172.44 │   2574.07 │
└───────────┴───────────┴──────────────┴───────────┘

In [14]:
duckdb.sql("""
    SELECT
        COUNT(*) AS zero_price_rows,
        COUNT(DISTINCT user_session) AS zero_price_sessions
    FROM 'february_cart_events.csv'
    WHERE price = 0
""")

┌─────────────────┬─────────────────────┐
│ zero_price_rows │ zero_price_sessions │
│      int64      │        int64        │
├─────────────────┼─────────────────────┤
│            1547 │                1219 │
└─────────────────┴─────────────────────┘

In [16]:
duckdb.sql("""
    WITH price_bands AS (
        SELECT
            user_session,
            purchased,
            CASE
                WHEN price <= 100 THEN '₹0-₹100'
                WHEN price <= 250 THEN '₹100-₹250'
                WHEN price <= 500 THEN '₹250-₹500'
                ELSE '₹500+'
            END AS price_band,

            CASE
                WHEN price <= 100 THEN 1
                WHEN price <= 250 THEN 2
                WHEN price <= 500 THEN 3
                ELSE 4
            END AS band_order

        FROM 'february_cart_events.csv'
    )

    SELECT
        price_band,
        COUNT(DISTINCT user_session) AS sessions,

        COUNT(DISTINCT CASE
            WHEN purchased = TRUE THEN user_session
        END) AS purchased_sessions,

        ROUND(
            100.0 * COUNT(DISTINCT CASE
                WHEN purchased = TRUE THEN user_session
            END)
            / COUNT(DISTINCT user_session),
            2
        ) AS purchase_rate

    FROM price_bands

    GROUP BY
        price_band,
        band_order

    ORDER BY band_order
""")

┌────────────┬──────────┬────────────────────┬───────────────┐
│ price_band │ sessions │ purchased_sessions │ purchase_rate │
│  varchar   │  int64   │       int64        │    double     │
├────────────┼──────────┼────────────────────┼───────────────┤
│ ₹0-₹100    │   542482 │             190025 │         35.03 │
│ ₹100-₹250  │   577423 │             277311 │         48.03 │
│ ₹250-₹500  │   294490 │             137580 │         46.72 │
│ ₹500+      │   316126 │             145449 │         46.01 │
└────────────┴──────────┴────────────────────┴───────────────┘

## Business Question 5: Does session duration relate to purchase behavior?

In [17]:
duckdb.sql("""
    SELECT
        MIN(session_duration_minutes) AS min_duration_minutes,
        ROUND(AVG(session_duration_minutes), 2) AS avg_duration_minutes,
        MEDIAN(session_duration_minutes) AS median_duration_minutes,
        MAX(session_duration_minutes) AS max_duration_minutes
    FROM (
        SELECT
            user_session,
            purchased,
            DATE_DIFF(
                'minute',
                MIN(event_time),
                MAX(event_time)
            ) AS session_duration_minutes
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    )
""")

┌──────────────────────┬──────────────────────┬─────────────────────────┬──────────────────────┐
│ min_duration_minutes │ avg_duration_minutes │ median_duration_minutes │ max_duration_minutes │
│        int64         │        double        │         double          │        int64         │
├──────────────────────┼──────────────────────┼─────────────────────────┼──────────────────────┤
│                    0 │                 3.46 │                     0.0 │                38978 │
└──────────────────────┴──────────────────────┴─────────────────────────┴──────────────────────┘

In [18]:
duckdb.sql("""
    SELECT
        purchased,
        COUNT(*) AS sessions,
        ROUND(AVG(session_duration_minutes), 2) AS avg_duration_minutes,
        MEDIAN(session_duration_minutes) AS median_duration_minutes
    FROM (
        SELECT
            user_session,
            purchased,
            DATE_DIFF(
                'minute',
                MIN(event_time),
                MAX(event_time)
            ) AS session_duration_minutes
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    )
    GROUP BY purchased
    ORDER BY purchased
""")

┌───────────┬──────────┬──────────────────────┬─────────────────────────┐
│ purchased │ sessions │ avg_duration_minutes │ median_duration_minutes │
│  boolean  │  int64   │        double        │         double          │
├───────────┼──────────┼──────────────────────┼─────────────────────────┤
│ false     │   960019 │                  3.8 │                     0.0 │
│ true      │   700091 │                 2.99 │                     0.0 │
└───────────┴──────────┴──────────────────────┴─────────────────────────┘

In [19]:
duckdb.sql("""
    SELECT
        user_session,
        COUNT(DISTINCT purchased) AS purchased_values
    FROM 'february_cart_events.csv'
    GROUP BY user_session
    HAVING COUNT(DISTINCT purchased) > 1
""")

┌──────────────┬──────────────────┐
│ user_session │ purchased_values │
│   varchar    │      int64       │
└──────────────┴──────────────────┘
              0 rows             

In [20]:
duckdb.sql("""
    WITH session_duration AS (
        SELECT
            user_session,
            purchased,
            DATE_DIFF(
                'minute',
                MIN(event_time),
                MAX(event_time)
            ) AS duration_minutes
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    ),

    duration_bands AS (
        SELECT
            user_session,
            purchased,

            CASE
                WHEN duration_minutes = 0 THEN '0 minutes'
                WHEN duration_minutes <= 2 THEN '1-2 minutes'
                WHEN duration_minutes <= 5 THEN '3-5 minutes'
                WHEN duration_minutes <= 10 THEN '6-10 minutes'
                ELSE '10+ minutes'
            END AS duration_band,

            CASE
                WHEN duration_minutes = 0 THEN 1
                WHEN duration_minutes <= 2 THEN 2
                WHEN duration_minutes <= 5 THEN 3
                WHEN duration_minutes <= 10 THEN 4
                ELSE 5
            END AS band_order

        FROM session_duration
    )

    SELECT
        duration_band,
        COUNT(*) AS sessions,

        SUM(
            CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
        ) AS purchased_sessions,

        ROUND(
            100.0 * SUM(
                CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS purchase_rate

    FROM duration_bands

    GROUP BY
        duration_band,
        band_order

    ORDER BY band_order
""")

┌───────────────┬──────────┬────────────────────┬───────────────┐
│ duration_band │ sessions │ purchased_sessions │ purchase_rate │
│    varchar    │  int64   │       int128       │    double     │
├───────────────┼──────────┼────────────────────┼───────────────┤
│ 0 minutes     │  1292378 │             466353 │         36.08 │
│ 1-2 minutes   │   171672 │              99929 │         58.21 │
│ 3-5 minutes   │    91104 │              62849 │         68.99 │
│ 6-10 minutes  │    52204 │              36604 │         70.12 │
│ 10+ minutes   │    52752 │              34356 │         65.13 │
└───────────────┴──────────┴────────────────────┴───────────────┘

## Business Question 6: Does the number of cart events in a session relate to purchase behavior?

In [21]:
duckdb.sql("""
    SELECT
        purchased,
        COUNT(*) AS sessions,
        ROUND(AVG(cart_events), 2) AS avg_cart_events,
        MEDIAN(cart_events) AS median_cart_events
    FROM (
        SELECT
            user_session,
            purchased,
            COUNT(*) AS cart_events
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    )
    GROUP BY purchased
    ORDER BY purchased
""")

┌───────────┬──────────┬─────────────────┬────────────────────┐
│ purchased │ sessions │ avg_cart_events │ median_cart_events │
│  boolean  │  int64   │     double      │       double       │
├───────────┼──────────┼─────────────────┼────────────────────┤
│ false     │   960019 │            1.42 │                1.0 │
│ true      │   700091 │            1.85 │                1.0 │
└───────────┴──────────┴─────────────────┴────────────────────┘

In [22]:
duckdb.sql("""
    WITH session_events AS (
        SELECT
            user_session,
            purchased,
            COUNT(*) AS cart_events
        FROM 'february_cart_events.csv'
        GROUP BY user_session, purchased
    ),

    event_bands AS (
        SELECT
            user_session,
            purchased,

            CASE
                WHEN cart_events = 1 THEN '1 event'
                WHEN cart_events = 2 THEN '2 events'
                WHEN cart_events = 3 THEN '3 events'
                ELSE '4+ events'
            END AS event_count_band,

            CASE
                WHEN cart_events = 1 THEN 1
                WHEN cart_events = 2 THEN 2
                WHEN cart_events = 3 THEN 3
                ELSE 4
            END AS band_order

        FROM session_events
    )

    SELECT
        event_count_band,
        COUNT(*) AS sessions,

        SUM(
            CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
        ) AS purchased_sessions,

        ROUND(
            100.0 * SUM(
                CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS purchase_rate

    FROM event_bands

    GROUP BY
        event_count_band,
        band_order

    ORDER BY band_order
""")

┌──────────────────┬──────────┬────────────────────┬───────────────┐
│ event_count_band │ sessions │ purchased_sessions │ purchase_rate │
│     varchar      │  int64   │       int128       │    double     │
├──────────────────┼──────────┼────────────────────┼───────────────┤
│ 1 event          │  1143622 │             404637 │         35.38 │
│ 2 events         │   308456 │             164996 │         53.49 │
│ 3 events         │   107012 │              65633 │         61.33 │
│ 4+ events        │   101020 │              64825 │         64.17 │
└──────────────────┴──────────┴────────────────────┴───────────────┘

## Business Question 7: Which product categories have higher or lower purchase rates?

In [23]:
duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(category_code) AS rows_with_category,
        COUNT(DISTINCT category_code) AS unique_categories
    FROM 'february_cart_events.csv'
""")

┌────────────┬────────────────────┬───────────────────┐
│ total_rows │ rows_with_category │ unique_categories │
│   int64    │       int64        │       int64       │
├────────────┼────────────────────┼───────────────────┤
│    2656529 │            2452815 │               137 │
└────────────┴────────────────────┴───────────────────┘

In [24]:
duckdb.sql("""
    SELECT
        category_code,
        COUNT(DISTINCT user_session) AS sessions,

        COUNT(DISTINCT CASE
            WHEN purchased = TRUE THEN user_session
        END) AS purchased_sessions,

        ROUND(
            100.0 * COUNT(DISTINCT CASE
                WHEN purchased = TRUE THEN user_session
            END)
            / COUNT(DISTINCT user_session),
            2
        ) AS purchase_rate

    FROM 'february_cart_events.csv'

    WHERE category_code IS NOT NULL

    GROUP BY category_code

    ORDER BY sessions DESC

    LIMIT 20
""")

┌───────────────────────────────────┬──────────┬────────────────────┬───────────────┐
│           category_code           │ sessions │ purchased_sessions │ purchase_rate │
│              varchar              │  int64   │       int64        │    double     │
├───────────────────────────────────┼──────────┼────────────────────┼───────────────┤
│ construction.tools.light          │   657112 │             344041 │         52.36 │
│ sport.bicycle                     │   118266 │              45779 │         38.71 │
│ electronics.clocks                │    63778 │              24832 │         38.94 │
│ apparel.shoes                     │    56099 │              20482 │         36.51 │
│ appliances.personal.massager      │    51696 │              23667 │         45.78 │
│ electronics.audio.headphone       │    37961 │              16573 │         43.66 │
│ appliances.environment.vacuum     │    31239 │              12925 │         41.37 │
│ appliances.kitchen.refrigerators  │    28848 │      

In [25]:
duckdb.sql("""
    WITH category_metrics AS (
        SELECT
            category_code,
            COUNT(DISTINCT user_session) AS sessions,

            COUNT(DISTINCT CASE
                WHEN purchased = TRUE THEN user_session
            END) AS purchased_sessions,

            ROUND(
                100.0 * COUNT(DISTINCT CASE
                    WHEN purchased = TRUE THEN user_session
                END)
                / COUNT(DISTINCT user_session),
                2
            ) AS purchase_rate

        FROM 'february_cart_events.csv'

        WHERE category_code IS NOT NULL

        GROUP BY category_code
    )

    SELECT
        category_code,
        sessions,
        purchased_sessions,
        purchase_rate
    FROM category_metrics
    WHERE sessions >= 10000
    ORDER BY purchase_rate DESC
""")

┌───────────────────────────────────┬──────────┬────────────────────┬───────────────┐
│           category_code           │ sessions │ purchased_sessions │ purchase_rate │
│              varchar              │  int64   │       int64        │    double     │
├───────────────────────────────────┼──────────┼────────────────────┼───────────────┤
│ appliances.kitchen.coffee_grinder │    13856 │               8399 │         60.62 │
│ construction.components.faucet    │    16345 │               8895 │         54.42 │
│ construction.tools.light          │   657112 │             344041 │         52.36 │
│ computers.peripherals.printer     │    23466 │              11497 │         48.99 │
│ appliances.personal.massager      │    51696 │              23667 │         45.78 │
│ appliances.kitchen.washer         │    24138 │              10830 │         44.87 │
│ furniture.bedroom.blanket         │    16664 │               7426 │         44.56 │
│ electronics.audio.headphone       │    37961 │      

## Business Question 8: Does brand relate to purchase behavior?

In [26]:
duckdb.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(brand) AS rows_with_brand,
        COUNT(DISTINCT brand) AS unique_brands
    FROM 'february_cart_events.csv'
""")

┌────────────┬─────────────────┬───────────────┐
│ total_rows │ rows_with_brand │ unique_brands │
│   int64    │      int64      │     int64     │
├────────────┼─────────────────┼───────────────┤
│    2656529 │         2399890 │          3383 │
└────────────┴─────────────────┴───────────────┘

In [27]:
duckdb.sql("""
    WITH brand_metrics AS (
        SELECT
            brand,
            COUNT(DISTINCT user_session) AS sessions,

            COUNT(DISTINCT CASE
                WHEN purchased = TRUE THEN user_session
            END) AS purchased_sessions,

            ROUND(
                100.0 * COUNT(DISTINCT CASE
                    WHEN purchased = TRUE THEN user_session
                END)
                / COUNT(DISTINCT user_session),
                2
            ) AS purchase_rate

        FROM 'february_cart_events.csv'

        WHERE brand IS NOT NULL

        GROUP BY brand
    )

    SELECT
        brand,
        sessions,
        purchased_sessions,
        purchase_rate

    FROM brand_metrics

    WHERE sessions >= 10000

    ORDER BY sessions DESC
""")

┌──────────┬──────────┬────────────────────┬───────────────┐
│  brand   │ sessions │ purchased_sessions │ purchase_rate │
│ varchar  │  int64   │       int64        │    double     │
├──────────┼──────────┼────────────────────┼───────────────┤
│ apple    │   353624 │             159422 │         45.08 │
│ samsung  │   305404 │             167146 │         54.73 │
│ xiaomi   │   154307 │              69134 │          44.8 │
│ huawei   │    47575 │              25761 │         54.15 │
│ lucente  │    25535 │              14528 │         56.89 │
│ oppo     │    19502 │              11546 │          59.2 │
│ lg       │    19366 │               8535 │         44.07 │
│ sony     │    18576 │               7246 │         39.01 │
│ acer     │    15373 │               7847 │         51.04 │
│ bosch    │    13825 │               5195 │         37.58 │
│ artel    │    13650 │               6450 │         47.25 │
│ lenovo   │    11023 │               5257 │         47.69 │
│ vitek    │    10888 │ 

## Business Question 9: Does purchase behavior vary by time of day?

In [28]:
duckdb.sql("""
    WITH session_hours AS (
        SELECT
            user_session,
            purchased,
            EXTRACT(
                HOUR FROM MIN(event_time)
            ) AS session_hour

        FROM 'february_cart_events.csv'

        GROUP BY
            user_session,
            purchased
    )

    SELECT
        session_hour,
        COUNT(*) AS sessions,

        SUM(
            CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
        ) AS purchased_sessions,

        ROUND(
            100.0 * SUM(
                CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS purchase_rate

    FROM session_hours

    GROUP BY session_hour

    ORDER BY session_hour
""")

┌──────────────┬──────────┬────────────────────┬───────────────┐
│ session_hour │ sessions │ purchased_sessions │ purchase_rate │
│    int64     │  int64   │       int128       │    double     │
├──────────────┼──────────┼────────────────────┼───────────────┤
│            0 │     8057 │               3007 │         37.32 │
│            1 │    14387 │               4838 │         33.63 │
│            2 │    29803 │              11139 │         37.38 │
│            3 │    57061 │              24084 │         42.21 │
│            4 │    82589 │              36137 │         43.76 │
│            5 │    98948 │              43359 │         43.82 │
│            6 │   109235 │              47954 │          43.9 │
│            7 │   113811 │              50059 │         43.98 │
│            8 │   116802 │              51541 │         44.13 │
│            9 │   115198 │              50742 │         44.05 │
│            · │      ·   │                ·   │           ·   │
│            · │      ·  

## Business Question 10: Does purchase behavior vary by day of the week?

In [29]:
duckdb.sql("""
    WITH session_days AS (
        SELECT
            user_session,
            purchased,
            DAYNAME(MIN(event_time)) AS day_name,
            DAYOFWEEK(MIN(event_time)) AS day_number

        FROM 'february_cart_events.csv'

        GROUP BY
            user_session,
            purchased
    )

    SELECT
        day_name,
        COUNT(*) AS sessions,

        SUM(
            CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
        ) AS purchased_sessions,

        ROUND(
            100.0 * SUM(
                CASE WHEN purchased = TRUE THEN 1 ELSE 0 END
            ) / COUNT(*),
            2
        ) AS purchase_rate

    FROM session_days

    GROUP BY
        day_name,
        day_number

    ORDER BY day_number
""")

┌───────────┬──────────┬────────────────────┬───────────────┐
│ day_name  │ sessions │ purchased_sessions │ purchase_rate │
│  varchar  │  int64   │       int128       │    double     │
├───────────┼──────────┼────────────────────┼───────────────┤
│ Sunday    │   258210 │             108314 │         41.95 │
│ Monday    │   229450 │              95396 │         41.58 │
│ Tuesday   │   223421 │              96403 │         43.15 │
│ Wednesday │   254275 │             114549 │         45.05 │
│ Thursday  │   194258 │              84632 │         43.57 │
│ Friday    │   201478 │              88138 │         43.75 │
│ Saturday  │   299018 │             112659 │         37.68 │
└───────────┴──────────┴────────────────────┴───────────────┘